In [ ]:
# Empty code cell required for "ruff" extension

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.config import data_path
from src.spark_utils import get_spark
from src.feature_models.shingles import evaluate_shingles_pairs
import pyspark.sql.functions as F
from src.utils import hex_to_bigint

%load_ext autoreload
%autoreload 2

In [ ]:
spark = get_spark()
spark

In [ ]:
from src.utils import hexchunk_to_bigint


data = [(["FFFFFFFFFFFFFF2F", "FFFFFFFFFFFFFF2F"],)]
df = spark.createDataFrame(data, ["hex_array"])
df = df.withColumn("result", F.transform(
        F.col("hex_array"),
        lambda hex_str: F.transform(
            F.filter(
                F.split(hex_str, "(?<=\\G.{16})"),
                lambda chunk: F.length(chunk) == 16   # scarta chunk vuoti/incompleti
            ),
            hexchunk_to_bigint
        )
))
df.show()

In [ ]:
df = spark.read.csv(
    str(data_path("near_duplicates_qqp_benchmark.csv")),
    header=True,
    # sep="\\t",
    sep=";",
    inferSchema=True,
)

df = df.withColumn("is_near_duplicate", F.when(F.col("is_near_duplicate_semantic") == 1, True).otherwise(False))
df = df.drop("qid1", "qid2", "is_near_duplicate_semantic", "is_near_duplicate_syntax")
df = df.withColumnsRenamed({
    "Id": "ID",
    "question1": "data1",
    "question2": "data2",
})

shingles_metrics = []
# The parameters to test are: window_size, fingerprint_size, K (threshold to determine if its a near-dup or not)
# for k in range(2, 32):
shingles_df = df.alias("shingles_df")
# shingles_metrics.append((k, evaluate_shingles_pairs(shingles_df, shingle_size=3, fingerprint_size=64, K=k, over_words=False)))
evaluate_shingles_pairs(shingles_df, shingle_size=3, fingerprint_size=256, K=22, over_words=False)

# shingles_df.printSchema()
shingles_metrics

In [ ]:
spark.stop()